# ReST

Label-free transferability estimation from the stable rank of the final two layers.
Uses 200 unlabeled target samples and no target labels.

1. Extract stable-rank records per (model, dataset)
2. Build the four ReST elements
3. Compute the ReST score
4. Evaluate with weighted Kendall correlation
5. Compare against LEEP, LogME and ETran

Configuration lives in `configs/cnn.yaml`; the implementation lives in the `rest` package.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd

from rest.config import load_config, record_path
from rest.data import set_seed
from rest.extract import calculate_transferability_scores
from rest.ground_truth import get_ground_truth
from rest.score import build_elements, rest_score, evaluate, weighted_kendall

## 1-2. Load models and extract the stable-rank records

In [ ]:
cfg = load_config("../configs/cnn.yaml")
set_seed(cfg.seed)
device = cfg.resolved_device()
ground_truth = get_ground_truth("cnn")
print("device:", device)

all_json = {}
for dataset_name in [cfg.source_dataset] + cfg.target_datasets:
    path = record_path(cfg.out_dir, dataset_name)
    if os.path.exists(path):
        all_json[dataset_name] = json.load(open(path))
        print(f"=== {dataset_name}: cached ({len(all_json[dataset_name])} models)")
        continue

    print(f"\n=== {dataset_name} ===")
    records = {}
    for model_name in cfg.model_hub:
        try:
            record = calculate_transferability_scores(
                model_name, dataset_name,
                num_samples=cfg.num_samples, sample_seed=cfg.sample_seed,
                device=device, batch_size=cfg.batch_size, max_feats=cfg.max_feats)
        except Exception as exc:
            print(f"  [skip] {model_name}: {exc}")
            continue
        if record is None:
            continue
        record["finetune_accuracy"] = ground_truth.get(dataset_name, {}).get(model_name)
        records[model_name] = record
        print(f"  {model_name:<14} pen_sr={record['penultimate_layer'][0]:.3f} "
              f"clf_sr={record['classifier_layer'][0]:.3f}")

    all_json[dataset_name] = records
    json.dump(records, open(path, "w"), indent=2)

## 3. The four ReST elements

In [ ]:
df = build_elements(all_json, cfg.source_dataset, cfg.target_datasets,
                    get_ground_truth("cnn"), use_clf=cfg.use_clf)
df.head(12)

## 4. ReST score

In [ ]:
df = rest_score(df, gamma=cfg.gamma)
df[["target dataset", "pre-trained model", "ReST", "fine-tune accuracy"]]

## 5. Weighted Kendall correlation with ground truth

In [ ]:
taus = evaluate(df, cfg.target_datasets)
print(f"ReST (gamma={cfg.gamma}) | source={cfg.source_dataset}")
print("-" * 34)
for dataset_name in cfg.target_datasets:
    print(f"  {dataset_name:<14}{taus[dataset_name]:>8.4f}")
print("-" * 34)
print(f"  {'MEAN':<14}{taus['MEAN']:>8.4f}")

## 6. Baselines

In [ ]:
from rest.baselines import clip_pseudo_labels, etran, extract_features_logits, leep, logme

LABEL_INDEPENDENT = True
n_samples = cfg.num_samples if LABEL_INDEPENDENT else None

rows = []
for dataset_name in cfg.target_datasets:
    pseudo = None
    for model_name in cfg.model_hub:
        features, logits, y_true, subset, dataset = extract_features_logits(
            model_name, dataset_name, n_samples, cfg.sample_seed,
            device=device, batch_size=cfg.batch_size, max_feats=cfg.max_feats)
        if LABEL_INDEPENDENT:
            if pseudo is None:
                pseudo = clip_pseudo_labels(subset, dataset.classes, device=device)
            y = pseudo
        else:
            y = y_true
        rows.append({
            "target dataset": dataset_name, "pre-trained model": model_name,
            "LEEP": leep(logits, y), "LogME": logme(features, y),
            "ETran": etran(features, logits, y),
            "fine-tune accuracy": ground_truth[dataset_name][model_name],
        })

bdf = pd.DataFrame(rows)
mode = "200 samples + CLIP pseudo-labels" if LABEL_INDEPENDENT else "full split + true labels"
print(f"Baselines ({mode})")
print("-" * 51)
print(f"{'dataset':<14}{'LEEP':>9}{'LogME':>9}{'ETran':>9}{'ReST':>9}")
for dataset_name, sub in bdf.groupby("target dataset"):
    print(f"{dataset_name:<14}" + "".join(
        f"{weighted_kendall(sub['fine-tune accuracy'], sub[m]):>9.3f}"
        for m in ["LEEP", "LogME", "ETran"]) + f"{taus[dataset_name]:>9.3f}")